# ✈️ TravelTide – Part 1: Data Cleaning & Exploratory Data Analysis

**Masterschool (MSIT) Data Analytics – Mastery Project**

This notebook cleans and explores the session-level cohort table (one row = one browsing session). It is the foundation for Part 2 (user-level features and segmentation) and Part 3 (K-Means / DBSCAN and perk mapping).

**Cohort (Elena's brief):** sessions from 2023-01-04 onward, users with more than 7 sessions in that window → **49,211 sessions from 5,998 users**.

**How the data gets here**
1. SQL is written and tested in VS Code / Beekeeper (`sql/JT_session_and_user_base.sql`).
2. Query results are exported as CSV into the project's Google Drive `data` folder.
3. This notebook loads the CSVs from Drive (works in Colab and VS Code) – no live database connection needed for the analysis.

**Notebook outline**
1. Setup & data loading
2. First inspection
3. Data cleaning (types, text formatting, missing values, hotel nights, cancellations, age)
4. Derived session/trip columns used later in the project
5. Exploratory data analysis
6. Export & key findings

## 1 – Setup & Data Loading

In [15]:
# Install gdown only when it is missing (Colab has it, VS Code may not)
import importlib.util, subprocess, sys #
if importlib.util.find_spec("gdown") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gdown"], check=True)

from pathlib import Path
import gdown
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")
PALETTE = ["#1F6F8B", "#99A8B2", "#E6A157", "#4C9F70", "#C8553D", "#6C5B7B"]
sns.set_palette(PALETTE)

### 1.1 Load the CSV export from Google Drive

The shared Drive `data` folder contains all CSV files for our data analysis. They are downloaded once into a local `data/` folder; later runs reuse the local copy.

*Note: the folder is shared as "Anyone with the link – Viewer" for the download to work.*

In [17]:
DRIVE_FOLDER_ID = "13JVW64-U2woCIdmQ1KAQeTyiKwlwaJdc"   # project folder on Google Drive (check the ID if the download fails)
DATA_DIR = Path("data")
SESSIONS_FILE = DATA_DIR/"JT_session_level_table.csv"

if not SESSIONS_FILE.exists():
    print("Downloading project data from Google Drive …")
    gdown.download_folder(id=DRIVE_FOLDER_ID, output=str(DATA_DIR), quiet=False)

def read_export(path):
    # SQL-client exports are sometimes Windows-1252 encoded (e.g. "AeroMéxico"), so fall back if UTF-8 fails
    try:
        return pd.read_csv(path)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp1252")

sessions_raw = read_export(SESSIONS_FILE)
df = sessions_raw.copy()
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]} | Users: {df['user_id'].nunique():,}")

Retrieving folder contents


Retrieving folder 1KgIwW9kSH8FLcT3pjmtYLnoTjsnj5z2F data
Processing file 1t70qKLUdLpVvXt64VtwvL07ybOE35NUQ flights.csv
Processing file 1gCfXJuu7U1fBDgVRtM8rFG6ywPLmGjN3 hotels.csv
Processing file 1cZGmTakhtkm0vKJlguPk5H_Ea0SjjCyM JT_ session_level_table.csv
Processing file 1QrpDezsdcdQZdn5uYO1PrgcM8ZCfSiQF sessions.csv
Processing file 1yJAQvVu8xd65pbMgfNcgPXH4IEoQ0ZcV users.csv
Retrieving folder 1gkTgfEA2z2N3S1rUcicZikILyzc-F1P4 notebooks
Retrieving folder 1VcAAbfCWkdwMIll5FycgfVwAdMdqcu1U SQL
Processing file 1HhSinoJj-1O-g58GYL26ZfuUhCc-moq9 filtered_tables.sql
Processing file 1AhJmKcAwmSCS_Zp9bvGD7zqdy2b4NOF3 JT_session_and_user_base.sql


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1t70qKLUdLpVvXt64VtwvL07ybOE35NUQ
To: c:\Users\jackl\OneDrive\Attachments\Desktop\Masterschool\Mastery Project\TravelTide Python\data\data\flights.csv
100%|██████████| 1.97M/1.97M [00:00<00:00, 13.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1gCfXJuu7U1fBDgVRtM8rFG6ywPLmGjN3
To: c:\Users\jackl\OneDrive\Attachments\Desktop\Masterschool\Mastery Project\TravelTide Python\data\data\hotels.csv
100%|██████████| 1.63M/1.63M [00:00<00:00, 15.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1cZGmTakhtkm0vKJlguPk5H_Ea0SjjCyM
To: c:\Users\jackl\OneDrive\Attachments\Desktop\Masterschool\Mastery Project\TravelTide Python\data\data\JT_ session_level_table.csv
100%|██████████| 792k/792k [00:00<00:00, 10.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1QrpDezsdcdQZdn5uYO1PrgcM8ZCfSiQF
To: c:\Users\jackl\OneDrive\

FileNotFoundError: [Errno 2] No such file or directory: 'data\\JT_session_level_table.csv'

### 1.2 Optional: live queries against the TravelTide database

The analysis runs on the exported CSV. A live connection is only needed for checks the cohort export **cannot** answer – for example, how the gender split looked **before** the cohort filter. Set `RUN_LIVE_QUERIES = True` to run them.

Store the connection URL as a Colab secret or environment variable named `TRAVELTIDE_URL` (never commit it to GitHub). SQLAlchemy uses `psycopg2` underneath.

In [19]:
RUN_LIVE_QUERIES = False

if RUN_LIVE_QUERIES:
    import os
    if importlib.util.find_spec("psycopg2") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "psycopg2-binary"], check=True)
    from sqlalchemy import create_engine, text
    try:
        from google.colab import userdata
        db_url = userdata.get("TRAVELTIDE_URL")
    except ImportError:
        db_url = os.environ["TRAVELTIDE_URL"]
    engine = create_engine(db_url.replace("postgres://", "postgresql://", 1))

    gender_by_stage = pd.read_sql(text('''
        WITH active AS (SELECT DISTINCT user_id FROM sessions WHERE session_start >= '2023-01-04'),
             cohort AS (SELECT user_id FROM sessions WHERE session_start >= '2023-01-04'
                        GROUP BY user_id HAVING COUNT(*) > 7),
             stages AS (
                 SELECT '1 all users' AS stage, gender FROM users
                 UNION ALL SELECT '2 active since 2023-01-04', u.gender FROM users u JOIN active a USING (user_id)
                 UNION ALL SELECT '3 cohort (>7 sessions)',   u.gender FROM users u JOIN cohort c USING (user_id))
        SELECT stage, gender, COUNT(*) AS users,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY stage), 1) AS pct
        FROM stages GROUP BY stage, gender ORDER BY stage, gender
    '''), engine)
    display(gender_by_stage)

## 2 – First Inspection

In [18]:
df.head(3).T

NameError: name 'df' is not defined

In [ ]:
df.info()

In [ ]:
# Cohort sanity checks: every user must have at least 8 sessions, all sessions inside the window
print("Min sessions per user:", df.groupby("user_id")["session_id"].count().min())
print("Session window:", df["session_start"].min(), "→", df["session_end"].max())
print("Duplicate session_ids:", df["session_id"].duplicated().sum(), "| Fully duplicated rows:", df.duplicated().sum())

**Observations:** 42 columns from four joined tables. Dates are stored as text, several integer columns are floats (because of missing values), and text columns use mixed casing. There are no duplicate sessions, and every user has at least 8 sessions, so the cohort filter worked as intended.

## 3 – Data Cleaning

**Guiding rule:** no session rows are dropped. Removing rows would change the cohort (users could fall below 8 sessions) and break the link to Elena's definition. Problems are fixed or flagged instead.

### 3.1 Date and time columns

In [ ]:
datetime_cols = ["session_start", "session_end", "departure_time", "return_time", "check_in_time", "check_out_time"]
date_cols = ["birthdate", "sign_up_date"]

# ISO8601 handles mixed precision (some check-in times carry microseconds)
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], format="ISO8601")
for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")

print("Unparsed values:", df[datetime_cols + date_cols].isna().sum().sum() - sessions_raw[datetime_cols + date_cols].isna().sum().sum())
print("session_end before session_start:", (df["session_end"] < df["session_start"]).sum())
print("return before departure:", (df["return_time"] < df["departure_time"]).sum())

### 3.2 Integer and boolean columns

`seats`, `checked_bags`, `rooms` and `nights` are counts, but pandas reads them as floats because sessions without a flight or hotel have no value. They become **nullable integers (`Int64`)** – missing stays missing.

> Filling them with 0 would be wrong: a browsing session did not book "0 seats", it booked no flight at all. Zeros would pull every average down.

In [ ]:
count_cols = ["seats", "checked_bags", "rooms", "nights"]
df[count_cols] = df[count_cols].astype("Int64")
df[count_cols].dtypes

`return_flight_booked` only applies to sessions with a flight. It is missing for all 34,891 sessions without a flight (correct) and for 50 flight sessions. Those 50 also have no `return_time`, so they are one-way flights → `False`. The column becomes a nullable boolean so "no flight" stays missing.

> Careful: `astype(bool)` on a column with `NaN` turns every `NaN` into `True`.

In [ ]:
print(pd.crosstab(df["flight_booked"], df["return_flight_booked"], dropna=False))
one_way = df["flight_booked"] & df["return_flight_booked"].isna()
print("Flight sessions without return info:", one_way.sum(), "| of these with a return_time:", df.loc[one_way, "return_time"].notna().sum())

df["return_flight_booked"] = df["return_flight_booked"].astype("boolean")
df.loc[one_way, "return_flight_booked"] = False
df["return_flight_booked"].value_counts(dropna=False)

### 3.3 Text formatting and unused columns

Text values are standardized for readable tables and charts (Tableau, Streamlit, slides):

| Column | Rule | Example |
|---|---|---|
| `home_country` | country name | `usa` → `USA`, `canada` → `Canada` |
| `home_city`, `destination` | Title Case | `new york` → `New York` |
| airport codes | upper case | `lga` → `LGA` |
| `hotel_name` | split into brand + city, Title Case, double spaces removed | `Banyan Tree  - oklahoma city` → brand `Banyan Tree` |
| `gender` | kept as `F` / `M` / `O` | – |

**Dropped because they are not used anywhere in the project:**
- `origin_airport` – identical to `home_airport` for 99.7 % of flights
- `trip_airline` – airlines play no role in the perks or segments
- hotel city part of `hotel_name` – matches `destination` for 98 % of package trips

In [ ]:
print("origin_airport == home_airport:", round((df.loc[df["flight_booked"], "origin_airport"] == df.loc[df["flight_booked"], "home_airport"]).mean() * 100, 1), "%")

for col in ["home_city", "destination"]:
    df[col] = df[col].str.strip().str.title()
for col in ["home_airport", "destination_airport"]:
    df[col] = df[col].str.strip().str.upper()
df["home_country"] = df["home_country"].str.strip().map({"usa": "USA", "canada": "Canada"})
df["gender"] = df["gender"].str.strip().str.upper()

# hotel_name = "Brand - city" → keep the brand only (city is already in destination)
df["hotel_brand"] = (df["hotel_name"].str.replace(r"\s+", " ", regex=True)
                     .str.split(" - ").str[0].str.strip().str.title()
                     .replace({"Nh Hotel": "NH Hotel", "Intercontinental": "InterContinental"}))   # keep brand spelling

df = df.drop(columns=["origin_airport", "trip_airline", "hotel_name"])
df[["home_country", "home_city", "home_airport", "destination", "destination_airport", "hotel_brand"]].dropna().head()

In [ ]:
for col in ["gender", "home_country"]:
    print(df.groupby("user_id")[col].first().value_counts().to_string(), "\n")
print("Hotel brands:", df["hotel_brand"].nunique(), "| Destinations:", df["destination"].nunique(), "| Home cities:", df["home_city"].nunique())

**Gender:** 88 % of cohort users are female, 12 % male and 11 users (0.2 %) `O`. The `O` users are **kept**: dropping them would remove sessions from the cohort, and gender is not used as a segmentation feature. The strong female majority is a property of the *filtered* cohort – see the live check in 1.2 for the split before filtering.

### 3.4 Missing values: structural or a problem?

Missing values in this table are expected when they belong to something that was not booked. The check below confirms that every gap is explained by the booking flags.

In [ ]:
flight_cols = ["destination", "destination_airport", "seats", "checked_bags", "departure_time",
               "destination_airport_lat", "destination_airport_lon", "base_fare_usd"]
hotel_cols = ["hotel_brand", "nights", "rooms", "check_in_time", "check_out_time", "hotel_per_room_usd"]

# Expected reason for a gap in each column group
expected_gap = {
    **{c: ~df["flight_booked"] | df["cancellation"] for c in flight_cols},
    **{c: ~df["hotel_booked"] | df["cancellation"] for c in hotel_cols},
    "return_time": ~df["flight_booked"] | df["cancellation"] | (df["return_flight_booked"] == False),
    "return_flight_booked": ~df["flight_booked"],
    "flight_discount_amount": ~df["flight_discount"] | df["cancellation"],
    "hotel_discount_amount": ~df["hotel_discount"] | df["cancellation"],
    "trip_id": ~df["flight_booked"] & ~df["hotel_booked"],
}
checks = df.isna().sum().to_frame("missing_total").query("missing_total > 0")
checks["explained"] = [(df[c].isna() & expected_gap[c]).sum() for c in checks.index]
checks["unexplained"] = checks["missing_total"] - checks["explained"]
checks.sort_values("missing_total", ascending=False)

- **Flight and hotel columns:** missing when no flight / hotel was booked → structural, no imputation. The few extra gaps (50 flight, 193 hotel) are all on **cancellation sessions**, which repeat the booking flags but not always the trip details – they are excluded from travel behavior anyway (3.6).
- **`return_time`:** additionally missing for the 668 one-way flights → correct.
- **`trip_id`:** missing for the 32,509 browsing-only sessions → correct.
- **Discount amounts:** missing when no discount was offered. 610 sessions have a discount flag but no amount – these are exactly the cancellation sessions (see 3.6). Note: a discount *flag* on a browsing session means a discount was **offered**, not used; "discount used" is only measured on booked trips (section 4).

In [ ]:
print("Discount flag without amount:", (df["flight_discount"] & df["flight_discount_amount"].isna()).sum(),
      "| all on cancellation sessions:", (df["flight_discount"] & df["flight_discount_amount"].isna() & df["cancellation"]).sum())
print("Flight discounts offered in sessions without a flight booking:",
      (df["flight_discount"] & ~df["flight_booked"]).sum(), "of", df["flight_discount"].sum())

### 3.5 Hotel nights

The raw `nights` column contains negative values and often disagrees with the check-in/check-out dates. Comparing it with the **calendar nights** (check-out date minus check-in date) shows the pattern: raw `nights` counts full 24-hour periods, so a stay from 14:47 to 11:00 the next morning shows 0 nights instead of 1.

**Decision:** use calendar nights (`nights_calc`). Stays with 0 or negative calendar nights (check-out on or before check-in) are invalid hotel dates → `nights_calc` is set to missing and flagged, the session itself is kept.

In [ ]:
calendar_nights = (df["check_out_time"].dt.normalize() - df["check_in_time"].dt.normalize()).dt.days
print(pd.crosstab(df["nights"].clip(-2, 4), calendar_nights.clip(-2, 4), rownames=["raw nights"], colnames=["calendar nights"]))

df["invalid_hotel_dates"] = calendar_nights.le(0) & df["hotel_booked"]
df["nights_calc"] = calendar_nights.where(calendar_nights > 0).astype("Int64")
print("\nRaw nights < 0:", (df["nights"] < 0).sum(),
      "| raw ≠ calendar:", (df["nights"] != calendar_nights)[df["hotel_booked"]].sum(),
      "| invalid hotel dates:", df["invalid_hotel_dates"].sum(),
      f"({df['invalid_hotel_dates'].sum() / df['hotel_booked'].sum():.1%} of hotel sessions)")
df = df.drop(columns="nights")

### 3.6 Cancellations

A cancellation is recorded as a **second session with the same `trip_id`** (`cancellation = True`, booking flags repeated). Counting both sessions would double-count the trip, and counting the original booking would treat a cancelled trip as travel.

**Decision:** flag every session whose trip was later cancelled (`trip_cancelled`). A **valid booking** is a booking session whose trip was never cancelled.

In [ ]:
cancelled_ids = set(df.loc[df["cancellation"], "trip_id"])
df["trip_cancelled"] = df["trip_id"].isin(cancelled_ids)
df["valid_booking"] = df["trip_id"].notna() & ~df["cancellation"] & ~df["trip_cancelled"]

print("Cancellation sessions:", df["cancellation"].sum(), "| cancelled trips:", len(cancelled_ids))
print("Sessions with a trip_id:", df["trip_id"].notna().sum(), "| unique trips:", df["trip_id"].nunique())
print("Valid (kept) bookings:", df["valid_booking"].sum())
print("Users without any valid booking:", (~df.groupby("user_id")["valid_booking"].any()).sum())

### 3.7 Age and account tenure

**Which date to measure age at?** All behavior in the cohort happens between 2023-01-04 and 2023-07-28. Age is a user attribute used at user level, so every user needs **one** age:

- **Reference date = last session end in the data (2023-07-28)** ✅ – same reference for everyone, works for the 556 users who never booked, results don't change when the notebook is re-run later.
- Today's date ❌ – adds 3+ years to everyone and changes every time the notebook runs.
- Age at each booking or trip ❌ – differs from the reference age by at most 7 months, is missing for non-bookers and gives several ages per user.

Age is computed in exact years (birthday reached or not), not `days // 365`, which drifts with leap years.

In [ ]:
REF_DATE = df["session_end"].max().normalize()
print("Reference date:", REF_DATE.date())

def age_on(birth, ref):
    # Exact age in full years: subtract one if the birthday hasn't happened yet in the reference year
    before_birthday = (birth.dt.month > ref.month) | ((birth.dt.month == ref.month) & (birth.dt.day > ref.day))
    return (ref.year - birth.dt.year - before_birthday.astype(int)).astype("Int64")

df["age"] = age_on(df["birthdate"], REF_DATE)
df["tenure_months"] = ((REF_DATE - df["sign_up_date"]).dt.days / 30.44).round(1)

users = df.groupby("user_id")[["age", "tenure_months"]].first()
print(users.describe().round(1))
print("\nUsers under 18 on the reference date:", (users["age"] < 18).sum())

**Data note:** 246 cohort users (4 %) are under 18 on the reference date (youngest: 16). They are **kept** – they are real active accounts – but flagged with `is_minor`. Recommendation for TravelTide: check the sign-up age rules; for the rewards program, perks for these users may need a guardian or adult account.

In [ ]:
df["is_minor"] = df["age"] < 18
df = df.drop(columns=["birthdate", "sign_up_date"])

### 3.8 Outliers

In [ ]:
outlier_cols = ["page_clicks", "base_fare_usd", "hotel_per_room_usd", "seats", "checked_bags", "rooms", "nights_calc"]

def iqr_summary(s):
    q1, q3 = s.quantile([.25, .75]); upper = q3 + 1.5 * (q3 - q1)
    return pd.Series({"median": s.median(), "q3": q3, "upper_fence": upper, "max": s.max(),
                      "share_above_fence_%": round((s > upper).mean() * 100, 1)})

pd.DataFrame({c: iqr_summary(df[c].dropna().astype(float)) for c in outlier_cols}).T.round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, title in zip(axes, ["base_fare_usd", "hotel_per_room_usd", "page_clicks"],
                          ["Flight base fare (USD)", "Hotel price per room-night (USD)", "Page clicks per session"]):
    sns.boxplot(x=df[col].dropna(), ax=ax, color=PALETTE[0], fliersize=2)
    ax.set_xscale("log"); ax.set_title(title, fontweight="bold"); ax.set_xlabel("log scale")
plt.tight_layout(); plt.show()

**Decision:** outliers are **kept**. They are plausible (long-haul business fares, luxury hotels, long research sessions) and some segments are defined by exactly these extremes (e.g. Long-haul Packers). Skewed features are log-transformed and scaled before clustering in Part 3 so extremes don't dominate the distance calculations.

## 4 – Derived Session & Trip Columns

Only columns that are used later in the project are created here. Each feeds a user-level feature in Part 2.

| New column | Definition | Used for |
|---|---|---|
| `session_minutes` | session end − start | engagement check |
| `booking_type` | browsing / flight + hotel / flight only / hotel only / cancellation | trip mix, Hotel Stayers |
| `trip_start`, `start_weekday`, `weekend_start` | earlier of departure and check-in; Fri/Sat start | Weekend Escapers |
| `lead_days` | days from booking session to trip start | flat-signal check |
| `distance_km` | haversine, home airport → destination airport | Long-haul Packers |
| `discount_used` | flight or hotel discount on a valid booking | Deal Hunters |
| `trip_spend_usd` | fare and hotel cost after discounts | value per segment |

In [ ]:
df["session_minutes"] = (df["session_end"] - df["session_start"]).dt.total_seconds() / 60

df["booking_type"] = np.select(
    [df["cancellation"],
     df["flight_booked"] & df["hotel_booked"],
     df["flight_booked"],
     df["hotel_booked"]],
    ["cancellation", "flight + hotel", "flight only", "hotel only"],
    default="browsing")

df["trip_start"] = df[["departure_time", "check_in_time"]].min(axis=1)
df["start_weekday"] = df["trip_start"].dt.day_name()
df["weekend_start"] = df["trip_start"].dt.dayofweek.isin([4, 5]).where(df["trip_start"].notna())   # Fri, Sat
df["lead_days"] = (df["trip_start"].dt.normalize() - df["session_end"].dt.normalize()).dt.days

def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine_km(df["home_airport_lat"], df["home_airport_lon"],
                                 df["destination_airport_lat"], df["destination_airport_lon"]).round(0)

df["discount_used"] = (df["flight_discount"] | df["hotel_discount"]).where(df["valid_booking"])
flight_cost = df["base_fare_usd"].fillna(0) * (1 - df["flight_discount_amount"].fillna(0))
hotel_cost = (df["hotel_per_room_usd"].fillna(0) * df["rooms"].fillna(0).astype(float)
              * df["nights_calc"].fillna(0).astype(float) * (1 - df["hotel_discount_amount"].fillna(0)))
df["trip_spend_usd"] = (flight_cost + hotel_cost).round(2).where(df["valid_booking"])

df["booking_type"].value_counts()

## 5 – Exploratory Data Analysis

In [ ]:
bookings = df[df["valid_booking"]].copy()          # one row per kept trip
users = df.groupby("user_id").first()              # one row per user (demographics)
print(f"Sessions: {len(df):,} | valid bookings: {len(bookings):,} | users: {len(users):,}")

### 5.1 Session activity over time

In [ ]:
monthly = df["session_start"].dt.to_period("M").value_counts().sort_index()
monthly.index = monthly.index.strftime("%b %Y")
ax = monthly.plot(kind="bar", color=PALETTE[0], figsize=(9, 4), rot=0)
ax.set_title("Sessions per month (cohort)", fontweight="bold"); ax.set_xlabel(""); ax.set_ylabel("Sessions")
plt.tight_layout(); plt.show()
print(monthly.to_string())

Session volume is more than twice as high in January–March as in April–July (July is a partial month, data ends 28 Jul). Part of this is likely seasonal trip planning; part may be how the cohort was built. Behavior is therefore summarized per user over the full window rather than compared month by month.

### 5.2 What happens in a session?

In [ ]:
order = ["browsing", "flight + hotel", "hotel only", "flight only", "cancellation"]
share = df["booking_type"].value_counts(normalize=True).reindex(order) * 100
ax = share.plot(kind="barh", color=PALETTE[0], figsize=(8, 3.5))
ax.invert_yaxis(); ax.xaxis.set_major_formatter(mtick.PercentFormatter())
for i, v in enumerate(share): ax.text(v + 0.5, i, f"{v:.1f}%", va="center")
ax.set_title("Session outcome", fontweight="bold"); ax.set_xlabel("Share of sessions")
plt.tight_layout(); plt.show()

About two thirds of all sessions end without a booking. Among bookings, flight + hotel packages dominate; hotel-only and flight-only trips are smaller but distinct groups. Cancellations are rare (1.2 % of sessions).

### 5.3 Who are the customers?

In [ ]:
users["family_status"] = np.select(
    [users["married"] & users["has_children"], users["married"], users["has_children"]],
    ["Married, children", "Married, no children", "Single, children"], default="Single, no children")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(users["age"].astype(float), bins=range(15, 90, 2), ax=axes[0], color=PALETTE[0])
axes[0].axvline(18, color=PALETTE[4], ls="--"); axes[0].text(18.5, axes[0].get_ylim()[1] * .9, "18", color=PALETTE[4])
axes[0].set_title("Age on 2023-07-28", fontweight="bold")
users["family_status"].value_counts().plot(kind="barh", ax=axes[1], color=PALETTE[0]); axes[1].set_title("Family status", fontweight="bold")
users["home_country"].value_counts().plot(kind="barh", ax=axes[2], color=PALETTE[0]); axes[2].set_title("Home country", fontweight="bold")
for ax in axes[1:]: ax.set_ylabel(""); ax.invert_yaxis()
plt.tight_layout(); plt.show()
print(users["family_status"].value_counts(normalize=True).round(3).to_string())

The typical customer is in her late 30s to 40s; 83 % live in the USA and 17 % in Canada. A third of users have children – relevant for a separate Family segment.

### 5.4 Who travels with whom? (party size and luggage)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for ax, col, title in zip(axes, ["seats", "rooms", "checked_bags"], ["Seats per flight", "Rooms per hotel stay", "Checked bags per flight"]):
    s = bookings[col].dropna().astype(int).value_counts(normalize=True).sort_index() * 100
    s.plot(kind="bar", ax=ax, color=PALETTE[0]); ax.set_title(title, fontweight="bold")
    ax.yaxis.set_major_formatter(mtick.PercentFormatter()); ax.set_xlabel(""); ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()

party = bookings.groupby("has_children")[["seats", "rooms", "checked_bags"]].mean().round(2)
party.index = ["No children", "Has children"]; party

Most trips are for one person with one room and at most one bag – about 12 % of flights have 2+ seats and 14 % of hotel stays 2+ rooms.

**Surprise:** users with children book almost the same party size as users without (1.20 vs 1.17 seats). `has_children` alone does **not** mean family travel – many parents travel alone. That's why the Family Crew rule needs both: children **and** 2+ seats or rooms on average.

### 5.5 When and how long? (weekend vs weekday, trip length)

In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
heat = (bookings.dropna(subset=["nights_calc"])
        .assign(nights=lambda d: d["nights_calc"].astype(int).clip(upper=8).astype(str).replace("8", "8+"))
        .pivot_table(index="start_weekday", columns="nights", values="trip_id", aggfunc="count")
        .reindex(days))
heat = heat[[c for c in [str(i) for i in range(1, 8)] + ["8+"] if c in heat.columns]]
plt.figure(figsize=(9, 4))
sns.heatmap(heat, cmap="Blues", annot=True, fmt=".0f", cbar=False)
plt.title("Hotel stays by start weekday and number of nights", fontweight="bold"); plt.xlabel("Nights"); plt.ylabel("")
plt.tight_layout(); plt.show()
print("Trips starting Fri/Sat:", f"{bookings['weekend_start'].mean():.1%}", "| median nights:", bookings["nights_calc"].median())

Short stays of 1–3 nights dominate. Overall, 28.7 % of trips start on a Friday or Saturday – exactly what you'd expect by chance (2 of 7 days = 28.6 %), so there is no general weekend bias. Weekend Escapers are therefore defined at **user** level: users whose own Fri/Sat share is clearly higher (≥ 50 %) and whose stays are short.

### 5.6 How far? (distance)

In [ ]:
ax = sns.histplot(bookings["distance_km"].dropna(), bins=60, color=PALETTE[0])
ax.axvline(3000, color=PALETTE[4], ls="--"); ax.text(3100, ax.get_ylim()[1] * .9, "3,000 km", color=PALETTE[4])
ax.set_title("Flight distance: home airport → destination", fontweight="bold"); ax.set_xlabel("km")
plt.tight_layout(); plt.show()
print("Flights over 3,000 km:", f"{(bookings['distance_km'] > 3000).mean():.1%}")

Most flights are domestic or cross-border (median about 2,000 km). A long tail of long-haul trips (> 3,000 km) – together with checked bags – defines the Long-haul Packers.

### 5.7 Discounts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bookings["discount_used"].map({True: "With discount", False: "Full price"}).value_counts().plot(
    kind="pie", autopct="%1.0f%%", ax=axes[0], colors=[PALETTE[1], PALETTE[0]], startangle=90)
axes[0].set_ylabel(""); axes[0].set_title("Valid bookings", fontweight="bold")
disc = pd.concat([bookings["flight_discount_amount"].dropna().rename("Flight"), bookings["hotel_discount_amount"].dropna().rename("Hotel")], axis=1)
disc_long = disc.melt(var_name="type", value_name="discount").dropna()
disc_long["discount"] = (disc_long["discount"].round(2) * 100).astype(int)
sns.countplot(data=disc_long, x="discount", hue="type", ax=axes[1])
axes[1].set_xticklabels([f"{int(float(l.get_text()))}%" for l in axes[1].get_xticklabels()])
axes[1].set_title("Discount size on booked trips", fontweight="bold"); axes[1].set_xlabel("Discount off the base price")
plt.tight_layout(); plt.show()

Roughly a quarter of bookings use a discount, mostly 5–20 %. Most customers book at full price – so an exclusive-discount perk should target the users who **consistently** book discounted trips (Deal Hunters), not everyone.

### 5.8 Flat signals – what *not* to segment on

In [ ]:
df["clicks_per_min"] = df["page_clicks"] / df["session_minutes"]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
sns.histplot(df["clicks_per_min"], bins=50, ax=axes[0], color=PALETTE[1]); axes[0].set_title("Clicks per minute", fontweight="bold")
sns.histplot(bookings["lead_days"].clip(upper=40), bins=40, ax=axes[1], color=PALETTE[1]); axes[1].set_title("Lead days (booking → trip), capped at 40", fontweight="bold")
df.groupby("user_id")["session_id"].count().value_counts().sort_index().plot(kind="bar", ax=axes[2], color=PALETTE[1])
axes[2].set_title("Sessions per user", fontweight="bold"); axes[2].tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
print(df["clicks_per_min"].describe().round(2).to_string())
print("\nLead days median:", bookings["lead_days"].median(), "| 90th percentile:", bookings["lead_days"].quantile(.9))
df = df.drop(columns="clicks_per_min")

- **Clicks per minute** is almost constant (~8 for everyone) → clicks and session length measure the same thing.
- **Lead time** is short for nearly everyone (median 8 days, 90 % within 12 days) → early planners vs last-minute bookers can't be separated.
- **Sessions per user** is 8 for most users because of the cohort rule.

These signals are documented but **not used** as segmentation features.

### 5.9 Correlation of trip features

In [ ]:
corr_cols = ["seats", "rooms", "checked_bags", "nights_calc", "distance_km", "base_fare_usd", "hotel_per_room_usd", "trip_spend_usd", "age"]
corr = bookings[corr_cols].astype(float).corr().round(2)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, mask=np.triu(np.ones_like(corr, dtype=bool)), annot=True, cmap="RdBu_r", center=0, vmin=-1, vmax=1, cbar=False)
plt.title("Correlation of trip features (valid bookings)", fontweight="bold"); plt.tight_layout(); plt.show()

- **Seats and rooms are almost the same signal (r = 0.98)** → in clustering, use one combined party-size feature instead of both, otherwise party size counts twice.
- Spend is driven by nights (0.55), fare (0.58) and party size; distance mainly works through the fare.
- Checked bags are only weakly linked to party size (0.15) → luggage is its own behavior (Long-haul Packers).
- **Age is unrelated to every trip feature (|r| ≤ 0.02)** → segment on **behavior**, and use demographics to describe the segments afterwards.

## 6 – Export & Key Findings

In [ ]:
df.to_csv(DATA_DIR / "sessions_clean.csv", index=False)
print("Saved:", DATA_DIR / "sessions_clean.csv", "|", df.shape)
df.dtypes.to_frame("dtype").T

Upload `data/sessions_clean.csv` to the Drive `data` folder so Part 2 and Tableau use the same cleaned data.

### Key findings
1. **Cohort:** 49,211 sessions from 5,998 users (Jan 4 – Jul 28, 2023); 88 % of users are female, 83 % live in the USA.
2. **Most sessions are browsing:** about two thirds of sessions end without a booking; 556 users (9 %) never kept a booking → a dedicated *Dreamers* segment.
3. **Data quality fixed, no rows dropped:** hotel nights recalculated from dates (raw values count 24-hour periods), 3 % invalid hotel dates flagged, 610 cancelled trips excluded from travel behavior, 246 under-18 users flagged.
4. **Strong behavioral signals:** party size (seats ≈ rooms), checked bags, trip length, distance, discount use (26 % of bookings) and hotel-only trips. Having children or being a certain age does **not** predict trip behavior on its own.
5. **Flat signals:** clicks per minute, lead time and sessions per user barely vary → not used for segmentation.

**Next:** Part 2 builds one row per user from these columns (`user_base` from SQL, checked against this table) and defines the segments.